In [1]:
import os
import random
import numpy as np
import torch


In [ ]:
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATASET_PATH = r"G:\Ragib\Prescription\Full_Dataset_Split"
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ColorJitter(contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


def is_valid_image(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class SafeImageFolder(datasets.ImageFolder):
    """ImageFolder that skips unreadable or zero-byte image files."""

    def __init__(self, root, transform=None, target_transform=None):
        super().__init__(root, transform=transform, target_transform=target_transform)
        valid_samples = []
        skipped = []
        for path, target in self.samples:
            if is_valid_image(path):
                valid_samples.append((path, target))
            else:
                skipped.append(path)
        if skipped:
            print(f"Skipping {len(skipped)} invalid image(s) in {root}:")
            for path in skipped:
                print(f"  - {path}")
        self.samples = valid_samples
        self.imgs = self.samples
        self.targets = [target for _, target in valid_samples]


train_ds = SafeImageFolder(os.path.join(DATASET_PATH, "Train"), train_transform)
val_ds   = SafeImageFolder(os.path.join(DATASET_PATH, "Val"), val_test_transform)
test_ds  = SafeImageFolder(os.path.join(DATASET_PATH, "Test"), val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(train_ds.classes)
print("Number of classes:", NUM_CLASSES)


Skipping 1 invalid image(s) in G:\Ragib\Prescription\Full_Dataset_Split\Test:
  - G:\Ragib\Prescription\Full_Dataset_Split\Test\Tufnil\38.png
Number of classes: 317


In [3]:
import torch.nn as nn
from torchvision import models

class AttentionFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 2),
            nn.Softmax(dim=1)
        )

    def forward(self, f1, f2):
        w = self.attn(torch.cat([f1, f2], dim=1))
        return w[:, 0:1] * f1 + w[:, 1:2] * f2


class FusionAttentionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.mobilenet = models.mobilenet_v2(pretrained=True)
        self.mobilenet.classifier = nn.Identity()

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()

        self.fusion = AttentionFusion(1280)

        self.classifier = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        f1 = self.mobilenet(x)
        f2 = self.efficientnet(x)
        fused = self.fusion(f1, f2)
        return self.classifier(fused)


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

_preview_model = FusionAttentionModel(NUM_CLASSES).to(DEVICE)
total_params, trainable_params = count_parameters(_preview_model)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nLayer-wise parameter count:")
for name, module in _preview_model.named_children():
    p = sum(x.numel() for x in module.parameters())
    print(f"{name}: {p:,}")

del _preview_model
torch.cuda.empty_cache()


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` 

Total parameters: 9,921,467
Trainable parameters: 9,921,467

Layer-wise parameter count:
mobilenet: 2,223,872
efficientnet: 4,007,548
fusion: 3,280,642
classifier: 409,405


In [5]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(loader):
    model.train()
    total_loss, correct = 0, 0

    for x, y in tqdm(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


def eval_epoch(loader):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


In [6]:
import json
import time
import random
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# =========================
# MULTI-RUN CONFIGURATION
# =========================
NUM_RUNS = 3
RANDOM_SEEDS = [42, 123, 456]
OUTPUT_DIR = "multirun_attention"
EPOCHS = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_run_results = []

for run_idx in range(NUM_RUNS):
    run_num = run_idx + 1
    seed = RANDOM_SEEDS[run_idx]
    run_dir = os.path.join(OUTPUT_DIR, f"run_{run_num}")
    os.makedirs(run_dir, exist_ok=True)

    complete_flag = os.path.join(run_dir, "run_complete.json")
    if os.path.exists(complete_flag):
        print(f"\n✅ Run {run_num} already complete. Loading saved results.")
        with open(complete_flag, "r", encoding="utf-8") as f:
            all_run_results.append(json.load(f))
        continue

    print(f"\n{'='*60}")
    print(f"Starting Run {run_num}/{NUM_RUNS} | Seed: {seed}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

    CHECKPOINT_PATH = os.path.join(run_dir, "checkpoint_attention_317.pth")
    BEST_MODEL_PATH = os.path.join(run_dir, "best_attention_317.pth")

    model = FusionAttentionModel(NUM_CLASSES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    start_epoch = 0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    if os.path.exists(CHECKPOINT_PATH):
        print("🔁 Resuming from checkpoint")
        ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        best_epoch = ckpt.get("best_epoch", start_epoch)

        train_losses = ckpt["train_losses"]
        val_losses   = ckpt["val_losses"]
        train_accs   = ckpt["train_accs"]
        val_accs     = ckpt["val_accs"]

    train_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        tr_loss, tr_acc = train_epoch(train_loader)
        val_loss, val_acc = eval_epoch(val_loader)

        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        train_accs.append(tr_acc)
        val_accs.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }, CHECKPOINT_PATH)

        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    training_time = time.time() - train_start

    plt.figure()
    plt.plot(train_accs, label="Train Acc")
    plt.plot(val_accs, label="Val Acc")
    plt.legend(); plt.grid()
    plt.savefig(os.path.join(run_dir, "accuracy_curve.png"), dpi=300)
    plt.close()

    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.legend(); plt.grid()
    plt.savefig(os.path.join(run_dir, "loss_curve.png"), dpi=300)
    plt.close()

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model.eval()

    y_true, y_pred = [], []
    y_pred_top5 = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            probs = torch.softmax(model(x), dim=1)
            preds = probs.argmax(1).cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(preds)
            y_pred_top5.extend(torch.topk(probs, 5, dim=1).indices.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )

    print("Precision (macro):", precision)
    print("Recall (macro):", recall)
    print("F1-score (macro):", f1)

    top1_acc = accuracy_score(y_true, y_pred)
    top5_acc = sum(
        y_true[i] in y_pred_top5[i]
        for i in range(len(y_true))
    ) / len(y_true)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12,10))
    sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
    plt.title("Confusion Matrix")
    plt.savefig(os.path.join(run_dir, "confusion_matrix.png"), dpi=300)
    plt.close()

    run_result = {
        "Run Number": run_num,
        "Random Seed": seed,
        "Top-1 Accuracy": top1_acc,
        "Top-5 Accuracy": top5_acc,
        "Precision": p_macro,
        "Recall": r_macro,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted,
        "Training Time": training_time,
        "Best Epoch": best_epoch,
    }
    all_run_results.append(run_result)

    with open(complete_flag, "w", encoding="utf-8") as f:
        json.dump(run_result, f, indent=2)

    with open(os.path.join(run_dir, "training_history.json"), "w", encoding="utf-8") as f:
        json.dump({
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }, f, indent=2)

    del model
    torch.cuda.empty_cache()

print(f"\nCompleted {len(all_run_results)} runs.")



Starting Run 1/3 | Seed: 42


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` 

🔁 Resuming from checkpoint
Precision (macro): 0.9908563045787021
Recall (macro): 0.9888156008029826
F1-score (macro): 0.9885340499429898


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` 


Starting Run 2/3 | Seed: 123


100%|██████████| 971/971 [01:55<00:00,  8.40it/s]


... (97 outputs left out)

100%|██████████| 971/971 [01:53<00:00,  8.54it/s]


Epoch 24/25
Train Loss: 1.0913 | Train Acc: 0.9959
Val   Loss: 1.1384 | Val   Acc: 0.9893


100%|██████████| 971/971 [02:37<00:00,  6.15it/s]


Epoch 25/25
Train Loss: 1.0824 | Train Acc: 0.9974
Val   Loss: 1.1025 | Val   Acc: 0.9924
Precision (macro): 0.9894887107078608
Recall (macro): 0.988242041869802
F1-score (macro): 0.9879412499903515

Completed 3 runs.


In [7]:
import pandas as pd
import numpy as np

metric_cols = [
    "Top-1 Accuracy",
    "Top-5 Accuracy",
    "Precision",
    "Recall",
    "Macro F1",
    "Weighted F1",
    "Training Time",
    "Best Epoch",
]

results_df = pd.DataFrame(all_run_results)
results_df = results_df[[
    "Run Number", "Random Seed",
    "Top-1 Accuracy", "Top-5 Accuracy",
    "Precision", "Recall", "Macro F1", "Weighted F1",
    "Training Time", "Best Epoch",
]]

summary_rows = []
for col in metric_cols:
    mean_val = results_df[col].mean()
    std_val = results_df[col].std(ddof=1) if len(results_df) > 1 else 0.0
    summary_rows.append({
        "Metric": col,
        "Mean ± Standard Deviation": f"{mean_val:.6f} ± {std_val:.6f}",
        "Mean": mean_val,
        "Standard Deviation": std_val,
    })

summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(OUTPUT_DIR, "summary_results.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "summary_results.xlsx")

with open(csv_path, "w", encoding="utf-8") as f:
    f.write("Per-Run Results\n")
    results_df.to_csv(f, index=False)
    f.write("\nMean ± Standard Deviation\n")
    summary_df[["Metric", "Mean ± Standard Deviation"]].to_csv(f, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Per-Run Results", index=False)
    summary_df[["Metric", "Mean ± Standard Deviation"]].to_excel(
        writer, sheet_name="Mean ± Std", index=False
    )

print("Saved:", csv_path)
print("Saved:", xlsx_path)
print("\nPer-Run Results:")
print(results_df.to_string(index=False))
print("\nMean ± Standard Deviation:")
print(summary_df[["Metric", "Mean ± Standard Deviation"]].to_string(index=False))


Saved: multirun_attention\summary_results.csv
Saved: multirun_attention\summary_results.xlsx

Per-Run Results:
 Run Number  Random Seed  Top-1 Accuracy  Top-5 Accuracy  Precision   Recall  Macro F1  Weighted F1  Training Time  Best Epoch
          1           42        0.988812        0.997992   0.990856 0.988816  0.988534     0.988531       0.000000          23
          2          123        0.990247        0.998566   0.990952 0.990249  0.990209     0.990220    3010.837033          25
          3          456        0.988239        0.998853   0.989489 0.988242  0.987941     0.987938    3080.675630          23

Mean ± Standard Deviation:
        Metric Mean ± Standard Deviation
Top-1 Accuracy       0.989099 ± 0.001034
Top-5 Accuracy       0.998470 ± 0.000438
     Precision       0.990432 ± 0.000819
        Recall       0.989102 ± 0.001034
      Macro F1       0.988895 ± 0.001176
   Weighted F1       0.988896 ± 0.001184
 Training Time 2030.504221 ± 1758.814914
    Best Epoch      23.66